# Polymarket Crypto Dataset Check

Lightweight validation notebook for the BTC/ETH core plus crypto policy/ETF/MicroStrategy dataset. This notebook does not run PCA, persistent homology, forecasting models, or neural nets.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"

markets = pd.read_parquet(PROCESSED / "market_universe.parquet")
selected = pd.read_csv(PROCESSED / "selected_markets.csv")
prices = pd.read_parquet(PROCESSED / "prices_long.parquet")
raw_panel = pd.read_parquet(PROCESSED / "panel_hourly_raw.parquet")
active_panel = pd.read_parquet(PROCESSED / "panel_hourly_active_ffill.parquet")
core_panel = pd.read_parquet(PROCESSED / "panel_hourly_core.parquet")
core_plus_panel = pd.read_parquet(PROCESSED / "panel_hourly_core_plus_satellites.parquet")
coverage_market = pd.read_csv(PROCESSED / "coverage_by_market.csv")
coverage_time = pd.read_csv(PROCESSED / "coverage_by_timestamp.csv")
validation = json.loads((PROCESSED / "validation_report.json").read_text())
manifest = json.loads((PROCESSED / "dataset_manifest.json").read_text())

prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True)
for panel in [raw_panel, active_panel, core_panel, core_plus_panel]:
    panel.index = pd.to_datetime(panel.index, utc=True)
    panel.columns = panel.columns.astype(str)

## Validation Summary

In [ ]:
print("analysis_ready:", validation["analysis_ready"])
print("failed gates:", [k for k, v in validation["quality_gates"].items() if not v])
print(json.dumps(validation["coverage"], indent=2))
print("limitations:")
for item in validation["limitations"]:
    print("-", item)

In [ ]:
pd.DataFrame({
    "gate": validation["quality_gates"].keys(),
    "passed": validation["quality_gates"].values(),
})

## Selected Families

In [ ]:
family_counts = selected["market_family"].value_counts().rename_axis("market_family").reset_index(name="markets")
family_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(family_counts["market_family"], family_counts["markets"], color="#2f6f73")
ax.set_title("Selected Markets by Family")
ax.set_ylabel("Markets")
ax.tick_params(axis="x", rotation=30)
plt.show()

## Active Markets Over Time

In [ ]:
coverage_time["timestamp"] = pd.to_datetime(coverage_time["timestamp"], utc=True)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(coverage_time["timestamp"], coverage_time["active_core_markets"], label="core", linewidth=1.5)
ax.plot(coverage_time["timestamp"], coverage_time["active_markets"], label="core + satellites", linewidth=1.0, alpha=0.8)
ax.axhline(validation["thresholds"]["min_median_active_core"], color="black", linestyle="--", linewidth=1, label="core gate")
ax.set_title("Active Markets Over Time")
ax.set_ylabel("Non-null markets")
ax.legend()
fig.autofmt_xdate()
plt.show()

## Missingness Heatmap

In [ ]:
heatmap_panel = core_plus_panel.copy()
if heatmap_panel.shape[1] > 60:
    keep_cols = heatmap_panel.notna().mean().sort_values(ascending=False).head(60).index
    heatmap_panel = heatmap_panel[keep_cols]
fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(heatmap_panel.notna().T, aspect="auto", interpolation="nearest", cmap="Greys")
ax.set_title("Core + Satellite Panel Observed Values")
ax.set_xlabel("Hourly timestamps")
ax.set_ylabel("Markets")
ax.set_yticks([])
plt.show()

## Coverage by Market

In [ ]:
coverage_market.sort_values("usable_points").head(10)[[
    "market_id", "question", "market_family", "asset", "usable_points", "observed_days", "timestamp_min", "timestamp_max"
]]

In [ ]:
coverage_market[["usable_points", "observed_days", "min_price", "max_price"]].describe()

## Panel Missingness

In [ ]:
pd.DataFrame({
    "panel": ["raw", "active_ffill", "core", "core_plus_satellites"],
    "rows": [raw_panel.shape[0], active_panel.shape[0], core_panel.shape[0], core_plus_panel.shape[0]],
    "markets": [raw_panel.shape[1], active_panel.shape[1], core_panel.shape[1], core_plus_panel.shape[1]],
    "missingness": [raw_panel.isna().mean().mean(), active_panel.isna().mean().mean(), core_panel.isna().mean().mean(), core_plus_panel.isna().mean().mean()],
    "timestamp_min": [raw_panel.index.min(), active_panel.index.min(), core_panel.index.min(), core_plus_panel.index.min()],
    "timestamp_max": [raw_panel.index.max(), active_panel.index.max(), core_panel.index.max(), core_plus_panel.index.max()],
})

## Sample BTC/ETH Histories

In [ ]:
sample_ids = (
    coverage_market[coverage_market["market_family"].isin(["btc_price", "eth_price"])]
    .sort_values("usable_points", ascending=False)
    .groupby("market_family")
    .head(3)["market_id"]
    .astype(str)
    .tolist()
)
plot_df = prices[prices["market_id"].astype(str).isin(sample_ids)].copy()
labels = selected.set_index(selected["market_id"].astype(str))["question"].to_dict()
fig, ax = plt.subplots(figsize=(12, 6))
for market_id, group in plot_df.groupby(plot_df["market_id"].astype(str)):
    group = group.sort_values("timestamp")
    ax.plot(group["timestamp"], group["yes_price"], label=labels.get(market_id, market_id)[:70])
ax.set_title("Sample BTC/ETH YES Price Histories")
ax.set_xlabel("Timestamp")
ax.set_ylabel("YES price")
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="best", fontsize=8)
fig.autofmt_xdate()
plt.show()

## Stress Tests

In [ ]:
stress_path = PROCESSED / "stress_tests" / "stress_test_summary.csv"
if stress_path.exists():
    stress = pd.read_csv(stress_path)
    display(stress)
else:
    print("Run python src/stress_test_dataset.py to create stress-test outputs.")